# RQ1 — Regressão Logística com as 58 features (protocolo do artigo) — Colab

Replica a linha **Logistic Regression da Tabela 3** do artigo (MAE 1,0149 ± 0,0043; RMSE 1,4691 ± 0,0042; AUC 0,7328 ± 0,0064) com o mesmo protocolo usado pelo SVM da réplica:

- `train.csv` + `test.csv` concatenados (o teste participa da validação cruzada — protocolo do artigo original);
- 5 folds estratificados por `categoria_rating`, `shuffle=False`;
- `RandomizedSearchCV` buscando RMSE, `n_iter=300` sobre o grid do artigo (50 combinações: `C` × `solver` × `class_weight`, com `max_iter=10000` e `random_state=None` — busca **exaustiva de 50 candidatos**);
- refit por fold + holdout `train.csv → test.csv`; matriz de confusão e JSON de artefatos.

**Código:** clona a branch `regressao-logistica-rq1` do repositório da equipe.
**Dados:** `train.csv` e `test.csv` não são versionados (licença não especificada); o notebook baixa automaticamente os splits do repositório original do artigo (mesma forma de obtenção documentada em `data/README.md`) ou, se preferir, você envia os dois CSVs manualmente na célula 2.

**Tempos no Colab gratuito (2 vCPU):** modo rápido ≈ 8–12 min; **execução completa ≈ 2–3 h** (deixe a aba aberta). No local, a busca levou ≈ 53 s por fit com 10 núcleos; cada linha `[CV k/5; i/50] END` no output é 1 dos 250 fits concluído

In [1]:
# Célula 1 — branch do experimento (clone) + dependências
import os, pathlib

# workers loky (joblib) imprimem progresso sem buffer -> linhas [CV] entram ao vivo no Colab
os.environ["PYTHONUNBUFFERED"] = "1"

if pathlib.Path("/content").exists() and not pathlib.Path("Projeto-IA-Replica-RatingPrediction").exists():
    !git clone -b regressao-logistica-rq1 https://github.com/Mur1love/Projeto-IA-Replica-RatingPrediction.git
    %cd Projeto-IA-Replica-RatingPrediction

# resolve a raiz do repo (funciona no COLab e no local/nbconvert — o cwd padrao do kernel
# do notebook e a pasta notebooks/, entao "." sozinho nao acha src/)
RAIZ = None
for cand in (pathlib.Path.cwd(), pathlib.Path.cwd().parent,
             pathlib.Path.cwd().parent / "Projeto-IA-Replica-RatingPrediction",
             pathlib.Path.cwd() / "Projeto-IA-Replica-RatingPrediction"):
    if (cand / "src" / "models" / "article_protocol.py").exists():
        RAIZ = cand
        break
assert RAIZ is not None, "raiz do repositório não encontrada (clone em célula anterior falhou?)"
os.chdir(RAIZ)
import sys
sys.path.insert(0, str(RAIZ))

try:
    import google.colab  # noqa: F401 — só para detectar Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # numpy/pandas continuam com as versões do Colab (fixá-las quebra o pacote do Colab);
    # a versão exata de cada biblioteca fica registrada nos JSON de saída (padrão do projeto).
    !pip install -q scikit-learn matplotlib joblib

import sklearn, numpy, pandas, matplotlib
print("IN_COLAB=%s | sklearn %s | numpy %s | pandas %s | raiz=%s" % (
    IN_COLAB, sklearn.__version__, numpy.__version__, pandas.__version__, RAIZ.name))


IN_COLAB=False | sklearn 1.9.1 | numpy 2.5.3 | pandas 3.0.6 | raiz=Projeto-IA-Replica-RatingPrediction


In [2]:
# Célula 2 — dados em data/raw/ (baixa automática do repositório original do artigo)
# alternativa manual: no Colab, rode upload_csv() e escolha train.csv e test.csv
import pathlib, zipfile, urllib.request

BASE_ZIP = "https://raw.githubusercontent.com/emanuellemarreira/rating-prediction-with-textual-features/main/dataset"
RAW = pathlib.Path("data/raw")
RAW.mkdir(parents=True, exist_ok=True)

SPLITS = {"train.csv": "train.zip", "test.csv": "test.zip"}


def baixar_split(nome_csv: str) -> pathlib.Path:
    destino = RAW / nome_csv
    if destino.exists():
        print("já existe:", destino)
        return destino
    zip_path = RAW / SPLITS[nome_csv]
    urllib.request.urlretrieve(f"{BASE_ZIP}/{SPLITS[nome_csv]}", zip_path)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(RAW)
    zip_path.unlink()
    print("baixado + extraído:", destino)
    return destino

def upload_csv_manual():
    from google.colab import files
    up = files.upload()  # escolha train.csv e depois test.csv
    for nome, conteudo in up.items():
        caminho = RAW / nome
        caminho.write_bytes(conteudo)
        print("upload salvo:", caminho)

_train = baixar_split("train.csv")
_test = baixar_split("test.csv")

# VALIDAÇÃO (data/README.md): splits oficiais, 61 colunas, 58 features, classes balanceadas
import pandas as pd
treino = pd.read_csv(_train); teste = pd.read_csv(_test)
assert list(treino.columns) == list(teste.columns), "colunas divergentes entre splits"
assert len(treino.columns) == 61, f"esperava 61 colunas, vejo {len(treino.columns)}"
assert len(treino) == 41172 and len(teste) == 10293, (len(treino), len(teste))
prefixos = ("str_", "pos_", "synt_", "lex_", "conc_", "twt_", "sbj_")
assert len([c for c in treino.columns if c.startswith(prefixos)]) == 58, "faltam features do artigo"
assert not treino.drop("text", axis=1).isna().any().any(), "valores ausentes nas features"
print("✔ base validada: %d treino + %d teste, 61 colunas (58 features), textos vazios: %d/%d" % (
    len(treino), len(teste), int(treino['text'].isna().sum()), int(teste['text'].isna().sum())))
print("rating (treino):", treino["rating"].value_counts().sort_index().to_dict())


já existe: data/raw/train.csv
já existe: data/raw/test.csv


✔ base validada: 41172 treino + 10293 teste, 61 colunas (58 features), textos vazios: 1/1
rating (treino): {1: 8234, 2: 8234, 3: 8235, 4: 8235, 5: 8234}


In [3]:
# Célula 3 — dados do protocolo (as 58 features; o teste entra na validação cruzada, como no artigo)
from src.models.article_protocol import load_article_data
from src.models.logistic_regression import LOGISTIC_REGRESSION_SPEC, PARAM_GRID

DADOS = load_article_data("data/raw/train.csv", "data/raw/test.csv")  # sample_n=None: base completa
n_grid = len(PARAM_GRID["C"]) * len(PARAM_GRID["solver"]) * len(PARAM_GRID["class_weight"])
print(f"{DADOS.n_rows} linhas ({DADOS.n_train_rows} train.csv + {DADOS.n_test_rows} test.csv), "
      f"{DADOS.n_features} features, texto vazio mantido: {DADOS.n_empty_text}")
print(f"grid do artigo: 5×5×2 = {n_grid} combinações -> com n_iter=300 a busca é exaustiva nas {n_grid}")


51465 linhas (41172 train.csv + 10293 test.csv), 58 features, texto vazio mantido: 2
grid do artigo: 5×5×2 = 50 combinações -> com n_iter=300 a busca é exaustiva nas 50


In [4]:
# Célula 4 (opcional) — modo rápido em amostra de 2.000 linhas: valida a pipeline antes das 2–3 h (~8–12 min)
RUN_QUICK = True   # pode deixar False depois de validar

if RUN_QUICK:
    from src.models.article_protocol import run_article_protocol, save_article_results
    dados_rapido = load_article_data("data/raw/train.csv", "data/raw/test.csv", sample_n=2000)
    resultado_rapido = run_article_protocol(
        LOGISTIC_REGRESSION_SPEC, dados_rapido, n_iter=300, verbose=2, n_jobs=-1)
    save_article_results(resultado_rapido, dados_rapido, "results/sample")  # ign. pelo Git
    print("\n=== RESUMO DO MODO RÁPIDO (amostra de 2.000 — não comparável à Tabela 3) ===")
    print("melhores hiperparâmetros da amostra:", resultado_rapido.best_params)
    for _m in ("mae", "rmse", "auc", "accuracy", "f1_macro"):
        _a = resultado_rapido.aggregate[_m]
        print(f"  {_m:8s} {_a['mean']:.4f} ± {_a['std']:.4f}  IC95% [{_a['ci95'][0]:.4f}, {_a['ci95'][1]:.4f}]")
    _h = resultado_rapido.holdout["metrics"]
    print(f"  holdout da amostra: MAE {_h['mae']:.4f} RMSE {_h['rmse']:.4f} AUC {_h['auc']:.4f} | "
          f"acurácia {_h['accuracy']:.4f} F1 {_h['f1_macro']:.4f}")
    print("artefatos em results/sample/ (ignorados pelo Git): metrics/logistic_regression_rq1*.json; results/sample/figures/")


Fitting 5 folds for each of 50 candidates, totalling 250 fits


/home/mleal/Workspace/Projeto-IA-Replica-RatingPrediction/.venv/lib64/python3.12/site-packages/sklearn/model_selection/_search.py:327: UserWarning: The total space of parameters 50 is smaller than n_iter=300. Running 50 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


[CV] END C=0.01, class_weight=None, max_iter=10000, random_state=None, solver=newton-cg; total time=   0.1s
[CV] END C=0.01, class_weight=None, max_iter=10000, random_state=None, solver=newton-cg; total time=   0.1s
[CV] END C=0.01, class_weight=None, max_iter=10000, random_state=None, solver=liblinear; total time=   0.0s
[CV] END C=0.01, class_weight=None, max_iter=10000, random_state=None, solver=liblinear; total time=   0.0s
[CV] END C=0.01, class_weight=None, max_iter=10000, random_state=None, solver=liblinear; total time=   0.0s
[CV] END C=0.01, class_weight=None, max_iter=10000, random_state=None, solver=liblinear; total time=   0.0s
[CV] END C=0.01, class_weight=None, max_iter=10000, random_state=None, solver=newton-cg; total time=   0.1s
[CV] END C=0.01, class_weight=None, max_iter=10000, random_state=None, solver=liblinear; total time=   0.0s
[CV] END C=0.01, class_weight=None, max_iter=10000, random_state=None, solver=newton-cg; total time=   0.1s


[CV] END C=0.01, class_weight=None, max_iter=10000, random_state=None, solver=lbfgs; total time=   0.4s
[CV] END C=0.01, class_weight=None, max_iter=10000, random_state=None, solver=newton-cg; total time=   0.1s
[CV] END C=0.01, class_weight=None, max_iter=10000, random_state=None, solver=lbfgs; total time=   0.4s
[CV] END C=0.01, class_weight=None, max_iter=10000, random_state=None, solver=lbfgs; total time=   0.5s


[CV] END C=0.01, class_weight=None, max_iter=10000, random_state=None, solver=lbfgs; total time=   0.5s
[CV] END C=0.01, class_weight=None, max_iter=10000, random_state=None, solver=lbfgs; total time=   0.5s


[CV] END C=0.01, class_weight=None, max_iter=10000, random_state=None, solver=sag; total time=   2.4s
[CV] END C=0.01, class_weight=None, max_iter=10000, random_state=None, solver=sag; total time=   2.5s
[CV] END C=0.01, class_weight=None, max_iter=10000, random_state=None, solver=sag; total time=   2.5s
[CV] END C=0.01, class_weight=balanced, max_iter=10000, random_state=None, solver=newton-cg; total time=   0.0s
[CV] END C=0.01, class_weight=balanced, max_iter=10000, random_state=None, solver=newton-cg; total time=   0.1s
[CV] END C=0.01, class_weight=balanced, max_iter=10000, random_state=None, solver=newton-cg; total time=   0.1s
[CV] END C=0.01, class_weight=None, max_iter=10000, random_state=None, solver=sag; total time=   2.3s
[CV] END C=0.01, class_weight=balanced, max_iter=10000, random_state=None, solver=newton-cg; total time=   0.1s
[CV] END C=0.01, class_weight=balanced, max_iter=10000, random_state=None, solver=newton-cg; total time=   0.0s


... (139 outputs left out)

[CV] END C=100, class_weight=balanced, max_iter=10000, random_state=None, solver=saga; total time=   7.4s
[CV] END C=100, class_weight=balanced, max_iter=10000, random_state=None, solver=saga; total time=   7.1s


/home/mleal/Workspace/Projeto-IA-Replica-RatingPrediction/.venv/lib64/python3.12/site-packages/sklearn/model_selection/_validation.py:493: FitFailedWarning: 
50 fits failed out of a total of 250.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
50 fits failed with the following error:
Traceback (most recent call last):
  File "/home/mleal/Workspace/Projeto-IA-Replica-RatingPrediction/.venv/lib64/python3.12/site-packages/sklearn/model_selection/_validation.py", line 855, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/mleal/Workspace/Projeto-IA-Replica-RatingPrediction/.venv/lib64/python3.12/site-packages/sklearn/base.py", line 1403, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^

[Parallel(n_jobs=-1)]: Done   3 out of   6 | elapsed:    0.5s remaining:    0.5s
[Parallel(n_jobs=-1)]: Done   6 out of   6 | elapsed:    0.5s finished



modo rápido concluído — resultados AMOSTRAIS, não comparáveis à Tabela 3


In [5]:
# Célula 5 — EXECUÇÃO COMPLETA: protocolo do artigo na base inteira (~2–3 h em Colab free)
# - cada linha "[CV k/5; i/50] END" no output indica 1 dos 250 fits concluídos (i é o candidato)
# - não feche a aba: sessões do Colab podem encerrar ao fechar/ao atingir limites de inatividade
# - em caso de queda, rode de novo do início (o protocolo não tem checkpoint intermediário)
# Após terminar, os JSONs/pimbem PNG são gravados em results/ (célula 7 copia para o Drive).
RUN_FULL = False    # <<< MUDE PARA True PARA A EXECUÇÃO REAL >>>

if RUN_FULL:
    assert "DADOS" in globals(), "rode a célula 3 antes"
if RUN_FULL:
    from src.models.article_protocol import run_article_protocol, save_article_results
    resultado = run_article_protocol(
        LOGISTIC_REGRESSION_SPEC, DADOS, n_iter=300, n_splits=5, seed=42, n_jobs=-1, verbose=2)
    caminhos = save_article_results(resultado, DADOS, "results")
    print("\n=== RESUMO DA EXECUÇÃO COMPLETA ===")
    print("melhores hiperparâmetros (réplica):", resultado.best_params)
    print("melhores do artigo:", LOGISTIC_REGRESSION_SPEC.article_best_params)
    for m in ("mae", "rmse", "auc", "accuracy", "f1_macro"):
        a = resultado.aggregate[m]
        ref = LOGISTIC_REGRESSION_SPEC.article_reference.get(m)
        extra = f"  (artigo: {ref[0]:.4f} ± {ref[1]:.4f})" if ref else ""
        print(f"{m:8s} {a['mean']:.4f} ± {a['std']:.4f}  IC95% [{a['ci95'][0]:.4f}, {a['ci95'][1]:.4f}]{extra}")
    h = resultado.holdout["metrics"]
    print(f"holdout (train.csv -> test.csv): MAE {h['mae']:.4f} RMSE {h['rmse']:.4f} "
          f"AUC {h['auc']:.4f} | acurácia {h['accuracy']:.4f} F1 {h['f1_macro']:.4f}")
    print("candidatos avaliados:", resultado.n_candidates,
          "| n_iter pedido:", resultado.config["n_iter_requested"])
    print("tempos (s):", resultado.timing)


In [6]:
# Célula 6 — comparação com a linha Logistic Regression da Tabela 3 do artigo
import json, pathlib
import pandas as pd

resumo_path = pathlib.Path("results/metrics/logistic_regression_rq1.json")
if resumo_path.exists():
    resumo = json.loads(resumo_path.read_text())
    comp = resumo["article"]["comparison"]
    tabelinha = pd.DataFrame(comp).T[["article_mean", "article_std", "replica_mean", "replica_std", "delta_mean"]]
    tabelinha.columns = ["artigo (média)", "artigo (desvio)", "réplica (média)", "réplica (desvio)", "Δ média"]
    display(tabelinha.round(4))
    fig = pathlib.Path("results/figures/confusion_matrix_logistic_regression.png")
    if fig.exists():
        from IPython.display import Image, display as mostrar
        mostrar(Image(filename=str(fig)))
else:
    print("execução completa ainda não gravada — rode a célula 5 com RUN_FULL=True")


execução completa ainda não gravada — rode a célula 5 com RUN_FULL=True


In [7]:
# Célula 7 — preservar os artefatos: copiar para o Drive e/ou baixar
USE_DRIVE = True     # mount pede permissão do Google (uma vez por sessão)

if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        import shutil
        destino = pathlib.Path("/content/drive/MyDrive/replica_rq1_logreg")
        destino.mkdir(parents=True, exist_ok=True)
        nomes = ("logistic_regression_rq1.json", "logistic_regression_rq1_folds.csv",
                 "logistic_regression_rq1_holdout.json")
        for nome in nomes:
            f = pathlib.Path("results/metrics") / nome
            if f.exists():
                shutil.copy(f, destino / nome)
        fig = pathlib.Path("results/figures/confusion_matrix_logistic_regression.png")
        if fig.exists():
            shutil.copy(fig, destino / fig.name)
        print("cópia do projeto no Drive:", destino)
    except Exception as exc:  # fora do Colab ou permissão negada: segue com o download
        print("Drive não disponível (%s); use o download direto abaixo" % type(exc).__name__)

try:
    from google.colab import files
    if resumo_path.exists():
        files.download(str(resumo_path))
        files.download("results/metrics/logistic_regression_rq1_folds.csv")
        files.download("results/figures/confusion_matrix_logistic_regression.png")
except (ImportError, NameError):
    print("fora do Colab (ou sem execução completa): artefatos já estão em results/")


Drive não disponível (ModuleNotFoundError); use o download direto abaixo
fora do Colab (ou sem execução completa): artefatos já estão em results/


## Depois de rodar (para o time)

1. **Copie para a issue/PR** (ou envie no chat): o `logistic_regression_rq1.json`, o `_folds.csv` e a matriz de confusão. Com os JSONs eu preencho as seções pendentes de `docs/diferencas.md` (nova seção da LR, §1, §8.1/8.2/8.5/8.6), `docs/documentacao.md` (§1.4, §10) e o `results/metrics/README.md`.
2. **O que faz o resultado ser comparável à Tabela 3:** base completa (5 folds sem shuffle, mesmo split do artigo), busca exaustiva nas 50 combinações com RMSE, seed 42. A contraparte TF-IDF dessa LR é o **baseline da Issue 7** (`baseline_logreg.json`) — a comparação TF-IDF × 58 features fecha com esse par.
3. **Limitações a registrar** (eu incluo na documentação com os números):
   - versões do Colab (scikit-learn/numpy) diferentes das travadas no `requirements.txt` — o JSON registra as exatas;
   - `liblinear` não resolve multiclasse no scikit-learn moderno (10 dos 50 candidatos falham com score NaN; a busca segue exaustiva nos 40 restantes e os melhores do artigo — `lbfgs, C=100` — continuam alcançáveis);
   - sem padronização das features (fidelidade ao artigo) e `max_iter=10000` do grid.
4. O modelo comparável fica em `results/metrics/` (base completa) — o modo rápido fica em `results/sample/` e **não** entra na documentação.